# RoboReplan — SFT warm-start + GRPO Training
**OpenEnv Hackathon submission**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jwalin-shah/robo-replan/blob/main/train/colab_train.ipynb) — *Open this link to always run the latest version from GitHub.*

Trains Qwen2.5 (0.5B or 1.5B) to plan, replan, and recover in a tabletop robot manipulation environment.

**Two-phase training:**
1. **SFT** on oracle rollouts — teaches the model the action space (fast, ~10 min)
2. **GRPO** with real env reward — teaches the model to maximize task success (RL, ~30 min)

**Key fix in GRPO reward**: each dataset row stores the serialized scenario config. The reward function reconstructs the exact env state per prompt before evaluating the model's action — so reward actually corresponds to the observation shown in the prompt.

**Run on:** Google Colab T4 (free tier; use 0.5B). For 1.5B use Colab Pro or more RAM; for H100 use the Northflank job or `bash train/run_h100_1.5b.sh`.  
**No Unsloth** — same stack as Northflank (TRL 0.14 + vllm) so it just runs.

In [ ]:
# Install dependencies and get latest code
# If you opened this notebook from GitHub (Open in Colab link above), we're already in the repo — pull latest.
# Otherwise we clone the Hugging Face space so you have the env and server code.
import os
if os.path.isdir('.git'):
    !git pull
else:
    !git clone https://huggingface.co/spaces/openenv-community/robo-replan
    %cd robo-replan

# Same stack as Northflank: trl 0.14 + vllm (no Unsloth — avoids GRPO crashes)
!pip install -q "trl==0.14.0" "vllm>=0.10.2,<0.13" transformers torch datasets openenv-core==0.2.1 pydantic numpy accelerate

# Ensure we're at repo root so "from server.*" works (Colab may start in train/ when opened from GitHub)
if os.path.basename(os.getcwd()) == 'train':
    os.chdir('..')

In [ ]:
# Verify environment
import sys
sys.path.insert(0, '.')

from server.config import EnvConfig
from server.environment import TabletopPlanningEnv

env = TabletopPlanningEnv(config=EnvConfig.easy())
obs = env.reset()
print('Environment OK')
print('Instruction:', obs.instruction)
print('Objects:', [(o.name, 'reachable' if o.reachable else 'BLOCKED') for o in obs.visible_objects])
print('Valid actions:', obs.valid_actions)
print('Oracle hint:', obs.oracle_hint)

In [ ]:
# ── Baseline: random policy BEFORE training ───────────────────────────
import random
from server.models import Action

ACTIONS = [a.value for a in Action]

def eval_policy(policy_fn, n_episodes=50):
    env = TabletopPlanningEnv(config=EnvConfig.easy())
    rewards, successes = [], []
    for _ in range(n_episodes):
        obs = env.reset()
        total = 0
        for _ in range(20):
            action = policy_fn(obs)
            r = env.step(action)
            total += r.reward
            obs = r.observation
            if r.done:
                break
        rewards.append(total)
        successes.append(env._all_goals_complete())
    return {
        'success_rate': sum(successes) / n_episodes,
        'avg_reward': sum(rewards) / n_episodes,
    }

baseline = eval_policy(lambda obs: random.choice(ACTIONS), n_episodes=50)
print(f'BEFORE TRAINING (random):')
print(f'  Success rate: {baseline["success_rate"]:.0%}')
print(f'  Avg reward:   {baseline["avg_reward"]:.2f}')

In [ ]:

# ── Build dataset from oracle rollouts ────────────────────────────────
# Each row stores the serialized scenario config so GRPO reward_fn
# can reconstruct the exact env state this prompt was generated from.
import json
from datasets import Dataset
from server.robosim.randomizer import ScenarioConfig

SYSTEM = (
    "You are a robot planning agent on a tabletop. Complete manipulation tasks "
    "by choosing ONE action per step.\n\n"
    "Actions: SCAN_SCENE | MOVE_TO_RED | MOVE_TO_BLUE | MOVE_TO_GREEN | "
    "MOVE_TO_YELLOW | MOVE_TO_PURPLE | PICK | PLACE_BIN_A | PLACE_BIN_B | CLEAR_BLOCKER\n\n"
    "Before each action, write a short plan inside <think>...</think>: "
    "list the remaining steps needed, then state what you are doing now and why.\n"
    "Example:\n"
    "<think>Plan: CLEAR_BLOCKER → MOVE_TO_RED → PICK → PLACE_BIN_A. "
    "Red is blocked so I clear first. Doing: CLEAR_BLOCKER.</think>\n"
    "CLEAR_BLOCKER"
)

def obs_to_user_msg(obs):
    objects = ', '.join(
        f"{o.name}({'reachable' if o.reachable else 'BLOCKED'})"
        for o in obs.visible_objects
    )
    history = ' -> '.join(obs.action_history[-5:]) or 'none'
    failures = '; '.join(obs.known_failures) or 'none'
    subgoals = '; '.join(obs.completed_subgoals) or 'none yet'
    valid = ', '.join(obs.valid_actions) if obs.valid_actions else 'any'
    return (
        f"Instruction: {obs.instruction}\n"
        f"Scene: {objects}\n"
        f"Holding: {obs.holding or 'nothing'}\n"
        f"Progress: {obs.goal_progress:.0%}  Remaining: {obs.goals_remaining}\n"
        f"Completed: {subgoals}\n"
        f"Failures: {failures}\n"
        f"History: {history}\n"
        f"Last: {obs.last_action or 'none'} -> {obs.last_result or 'n/a'}\n"
        f"Valid now: {valid}\n"
        f"Steps left: {obs.steps_remaining}\n\nNext action:"
    )

def scenario_to_json(scen) -> str:
    return json.dumps({
        'objects':       scen.objects,
        'targets':       scen.targets,
        'blockers':      scen.blockers,
        'distractors':   scen.distractors,
        'constraint':    scen.constraint,
        'instruction':   scen.instruction,
        'positions':     {k: list(v) for k, v in scen.positions.items()},
        'hidden_traits': scen.hidden_traits or {},
        'deadlines':     scen.deadlines or {},
    })

cfg = EnvConfig.easy()
cfg.obs.include_oracle_hint = True
env = TabletopPlanningEnv(config=cfg)

rows = []
N_EPISODES = 400  # 400 = faster SFT (~10–15 min); use 800 for better quality
for _ in range(N_EPISODES):
    obs = env.reset()
    scenario_json = scenario_to_json(env._scenario_cfg)
    step_num = 0
    for _ in range(20):
        if obs.oracle_hint:
            rows.append({
                'prompt': [
                    {'role': 'system', 'content': SYSTEM},
                    {'role': 'user',   'content': obs_to_user_msg(obs)},
                ],
                'answer':        obs.oracle_hint,
                'scenario':      scenario_json,
                'step':          step_num,
                'action_history': list(obs.action_history),  # replay in reward_fn to get correct state
            })
        action = obs.oracle_hint or 'SCAN_SCENE'
        r = env.step(action)
        obs = r.observation
        step_num += 1
        if r.done:
            break

dataset = Dataset.from_list(rows).train_test_split(test_size=0.1)
print(f'Dataset: {len(rows)} steps  ({len(dataset["train"])} train, {len(dataset["test"])} eval)')
print('Example action:', rows[0]['answer'])


In [ ]:
# ── Load model ────────────────────────────────────────────────────────
# Use 0.5B for free Colab T4; use 1.5B if you have Colab Pro / more RAM
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'  # or 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'Loading {MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map='auto'
)
print('Loaded on:', next(model.parameters()).device)

In [ ]:
# ── Phase 1: SFT warm-start ───────────────────────────────────────────
# Teaches the model the action space before GRPO.
# Without warm-start, GRPO starts from gibberish and wastes half its budget.
try:
    from trl import SFTTrainer, SFTConfig
except ModuleNotFoundError:
    !pip install -q "trl==0.14.0" "vllm>=0.10.2,<0.13" transformers torch datasets openenv-core==0.2.1 pydantic numpy accelerate
    from trl import SFTTrainer, SFTConfig

# Build columns for SFT: TRL expects 'prompt' + 'completion' (conversational format)
def add_messages_col(row):
    return {
        'messages': row['prompt'] + [{'role': 'assistant', 'content': row['answer']}],
        'completion': [{'role': 'assistant', 'content': row['answer']}],
    }

sft_train = dataset['train'].map(add_messages_col)
sft_eval  = dataset['test'].map(add_messages_col)

sft_config = SFTConfig(
    output_dir='./outputs/sft',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    max_seq_length=384,
    bf16=True,
    logging_steps=20,
    save_steps=200,
    eval_strategy='steps',
    eval_steps=200,
    report_to='none',
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_train,
    eval_dataset=sft_eval,
    processing_class=tokenizer,
)

print('Phase 1: SFT warm-start...')
sft_trainer.train()
sft_trainer.save_model('./outputs/sft_final')
print('SFT done -> ./outputs/sft_final')

In [ ]:
# ── Phase 2: GRPO with correct per-prompt env reward ──────────────────
# The reward function:
#   1. Deserializes the scenario config stored in the dataset row
#   2. Rebuilds the exact env state that generated this prompt
#   3. Executes the model's action in that state
#   4. Returns the real env reward (task reward + reasoning bonus)
#
# This is the correct approach — reward is causally linked to the prompt.
import re
import json
from trl import GRPOTrainer, GRPOConfig
from server.robosim.randomizer import ScenarioConfig

def extract_action(text: str):
    """Parse action, stripping <think> tags first."""
    clean = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip().upper()
    for a in sorted(ACTIONS, key=len, reverse=True):  # longest first avoids partial matches
        if a in clean:
            return a
    return None

def extract_reasoning(text: str) -> str:
    m = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    return m.group(1).strip() if m else ''

def json_to_scenario(s: str) -> ScenarioConfig:
    d = json.loads(s)
    d['positions'] = {k: tuple(v) for k, v in d['positions'].items()}
    return ScenarioConfig(**d)

def reward_fn(completions, prompts=None, scenario=None, action_history=None, **kwargs):
    rewards = []
    for i, completion in enumerate(completions):
        text = completion if isinstance(completion, str) else completion[0].get('content', '')
        action    = extract_action(text)
        reasoning = extract_reasoning(text)

        if action is None:
            rewards.append(-2.0)
            continue

        try:
            # Reconstruct exact env state for this prompt.
            # We build from the initial scenario config, then REPLAY all actions that
            # happened before this prompt was generated — so the env state matches the
            # observation shown in the prompt (e.g. holding an object at step 5).
            eval_env = TabletopPlanningEnv(config=EnvConfig.easy())
            scen = json_to_scenario(scenario[i])
            eval_env.sim._build_state_from_config(scen)
            eval_env._scenario_cfg          = scen
            eval_env._instruction           = scen.instruction
            eval_env._required_placements   = dict(scen.targets)
            eval_env._active_constraints    = [scen.constraint] if scen.constraint else []

            # Replay prior actions to reach the exact state this prompt was generated from
            history = action_history[i] if action_history is not None else []
            for past_action in (history or []):
                r = eval_env.step(past_action)
                if r.done:
                    break  # episode ended during replay — reward signal invalid

            result = eval_env.step(action, reasoning=reasoning)
            rewards.append(result.reward)
        except Exception:
            rewards.append(-1.0)

    return rewards

# Use SFT checkpoint as GRPO starting point
grpo_model = AutoModelForCausalLM.from_pretrained(
    './outputs/sft_final', torch_dtype=torch.bfloat16, device_map='auto'
)
grpo_model.warnings_issued = {}  # required by TRL 0.14 GRPOTrainer

grpo_config = GRPOConfig(
    output_dir='./outputs/grpo',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    num_generations=4,
    max_completion_length=64,  # enough room for <think>...</think> + action
    temperature=0.9,
    bf16=True,
    logging_steps=5,
    save_steps=50,
    report_to='none',
    use_vllm=False,  # use HF generation (vllm still required for TRL import)
)

trainer = GRPOTrainer(
    model=grpo_model,
    args=grpo_config,
    reward_funcs=reward_fn,
    train_dataset=dataset['train'],
    processing_class=tokenizer,
)

print('Phase 2: GRPO training...')
trainer.train()
trainer.save_model('./outputs/grpo_final')
print('GRPO done -> ./outputs/grpo_final')

In [ ]:
# ── Evaluate trained model ────────────────────────────────────────────
from transformers import pipeline

pipe = pipeline(
    'text-generation', model='./outputs/grpo_final',
    tokenizer=tokenizer, max_new_tokens=64, device_map='auto'
)

def trained_policy(obs):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user',   'content': obs_to_user_msg(obs)},
    ]
    out = pipe(messages, return_full_text=False)[0]['generated_text']
    return extract_action(out) or random.choice(ACTIONS)

after = eval_policy(trained_policy, n_episodes=50)
print(f'AFTER TRAINING (SFT + GRPO):')
print(f'  Success rate: {after["success_rate"]:.0%}')
print(f'  Avg reward:   {after["avg_reward"]:.2f}')
print()
print(f'Improvement:')
print(f'  Success: {baseline["success_rate"]:.0%} -> {after["success_rate"]:.0%}')
print(f'  Reward:  {baseline["avg_reward"]:.1f} -> {after["avg_reward"]:.1f}')

In [ ]:
# ── Plot results ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = ['Before\n(random)', 'After\n(SFT+GRPO)']
colors = ['#e05555', '#55c055']

axes[0].bar(labels, [baseline['success_rate'], after['success_rate']], color=colors)
axes[0].set_ylabel('Success Rate')
axes[0].set_title('Task Completion')
axes[0].set_ylim(0, 1)
for i, v in enumerate([baseline['success_rate'], after['success_rate']]):
    axes[0].text(i, v + 0.02, f'{v:.0%}', ha='center', fontweight='bold', fontsize=14)

axes[1].bar(labels, [baseline['avg_reward'], after['avg_reward']], color=colors)
axes[1].set_ylabel('Average Reward per Episode')
axes[1].set_title('Reward Improvement')
for i, v in enumerate([baseline['avg_reward'], after['avg_reward']]):
    axes[1].text(i, max(v, 0) + 0.2, f'{v:.1f}', ha='center', fontweight='bold', fontsize=14)

plt.suptitle('RoboReplan: SFT + GRPO Training Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_results.png')

In [ ]:
# ── Optional: push model to HF Hub ───────────────────────────────────
# trainer.model.push_to_hub('jshah13/robo-replan-grpo')
# tokenizer.push_to_hub('jshah13/robo-replan-grpo')
# print('https://huggingface.co/jshah13/robo-replan-grpo')